# ML baseline (logistic regression + random forest)

**CPU, no GPU benefit** -- sklearn LogisticRegression/RandomForest at this feature/row count are genuinely CPU-bound; a Colab GPU runtime typically grants FEWER vCPUs, so forcing one here would likely be slower, not faster.

Two classical ML models, trained against a binary `any_risk` target (`risk_factor_count > 0` -- the same ground truth RBAC/ABAC were already judged against). Each model's `P(any_risk=0)` becomes its continuous [0,1] trust score, fed through the SAME `decision_engine.trust_weighted_decision()` every other baseline uses -- so it lands in the identical PERMIT / CONDITIONAL-PERMIT / DENY decision space, not a separate binary one.

**Now variant-aware: trained ONCE on `main`, evaluated zero-shot everywhere else.** Mirrors the reference ACCESS-IoT dataset's own Table 5 precedent, rather than retraining 7 separate models or pooling all 7 builds into one training set: each model learns from `main`'s train split only, then the SAME frozen model is scored against every build's OWN test split (`main`, `sparse_feedback`, `noisy_benign`, `adversarial`, `seed_101`, `seed_103`, `seed_104`). That's the right question for this benchmark -- "does a model trained under ordinary conditions hold up when the audit trail gets quieter, honest behavior gets noisier, or the attacker camouflages itself" -- not "can a model be retrained to beat each variant."

**Features**: real session attributes and engineered columns already in the benchmark (billing_deviation, latency_deviation, previous_flag_rate, baseline_charge_duration_h, energy_rate_anomaly, raw telemetry) plus one new causal feature (`ev_session_share_so_far`). Deliberately EXCLUDES every RBAC/ABAC/naive-trust output column, AND cpo_id/emsp_id. The identity exclusion is a fix, not an original design choice: an earlier version of this notebook included cpo_id/emsp_id and got AUROC/AUPRC = 1.0000 exactly on the real data -- in this dataset the malicious CPO/eMSP are malicious from day 0 with no time variation, so the name alone perfectly determines the label, and a model with that as input just memorizes a lookup table instead of learning anything behavioral. See `ml_features.py`'s bug log for the full account.

**A second bug, found and fixed while wiring up the 7-build variant suite (read this if you ran an earlier version of this notebook):** `ev_session_share_so_far`'s running count/denominator used to be kept across the ENTIRE passed-in dataframe, with no awareness of which world/variant a row came from -- the same class of leak as `naive_trust.py`'s (see `04_rbac_abac_baselines.ipynb`'s intro): world 0 and world 1 (and now all 7 variants) share the SAME real connection_time values, so the running statistic silently carried over between what should be independent synthetic timelines. **This was already true for the original 2-world `main` dataset.** Fixed by grouping the running computation by `(variant, world_id)` -- see `ml_features.py`'s SECOND BUG LOG entry.

**CPU, sequential, not a multiprocessing Pool** -- a local smoke test on synthetic data caught a real bug in the Pool-of-models design this started with (RandomForest's own `n_jobs=-1` gets silently nerfed to 1 when nested inside a Pool worker, since Pool workers are daemonic and can't spawn children). Training sequentially and letting RandomForest's `n_jobs=-1` run directly in the main process is the actual higher-impact lever here -- see `ml_baseline.py`'s docstring.

**Checkpointed**: each model's `.joblib` (trained on `main` only) is saved the moment training finishes; re-running skips training for any model already on disk. The per-variant evaluation table is cheap (scoring, not training) and is always recomputed.

**Requires:** `v2g_benchmark_multi_world.csv` in your Drive (loads the combined file directly, not the pre-split train/test CSVs -- `ev_session_share_so_far` is causal and must be computed over each (variant, world_id) group's full timeline before splitting; see `ml_features.py`).

## Step 1: write the shared modules (decision layer, labels, split, features)

In [ ]:
%%writefile decision_engine.py
"""
decision_engine.py
-------------------
Implements Algorithm 4 from the TrustMesh framework doc (Section 8):
the trust-weighted three-way access decision (PERMIT / CONDITIONAL-PERMIT / DENY).

This is what turns a trust score into an actual access-control outcome,
which is the concrete instantiation of Gap 4.3.1 (fuzzy trust feeding
MA-ABAC decisions directly, not sitting beside them).
"""

from enum import Enum


class Decision(Enum):
    PERMIT = "PERMIT"
    CONDITIONAL_PERMIT = "CONDITIONAL-PERMIT"
    DENY = "DENY"


def trust_weighted_decision(base_confidence: float, effective_trust: float,
                             theta_high: float = 0.75, theta_low: float = 0.4) -> Decision:
    """
    effective_confidence = base_confidence * effective_trust   (framework doc Section 8)

    base_confidence: how strongly the attribute itself is asserted (e.g., a
    validly-signed certificate = 1.0; a partially-expired one might be lower).
    effective_trust: E[opinion] = b + a*u for the issuing authority.
    """
    effective_confidence = base_confidence * effective_trust
    if effective_confidence >= theta_high:
        return Decision.PERMIT
    elif effective_confidence >= theta_low:
        return Decision.CONDITIONAL_PERMIT
    else:
        return Decision.DENY


# ---------------------------------------------------------------------------
# Multi-attribute ABAC policy (combines authority_trust with OTHER
# attributes via rules, not just a single trust threshold -- this is what
# makes it attribute-based access control rather than trust-gating alone).
# ---------------------------------------------------------------------------

ROAMING_TRUST_PENALTY = 0.10  # roaming sessions apply a direct penalty to the
                                # CPO's trust value -- a real, common pattern
                                # (foreign/roaming access needs stronger
                                # assurance than home-network access)


def evaluate_access_policy(cpo_trust: float, emsp_trust: float, certificate_valid: bool,
                             is_roaming: bool, is_peak_hour: bool,
                             theta_high: float = 0.75, theta_low: float = 0.4,
                             roaming_penalty: float = ROAMING_TRUST_PENALTY) -> Decision:
    """
    GENUINE multi-authority ABAC policy: the decision is a function of TWO
    independently-computed authorities' trust scores, not one authority's
    trust plus trust-blind attribute checks.

      - cpo_trust:  TrustMesh's trust in the CPO (billing honesty)
      - emsp_trust: the hub-mediated, EFFECTIVE confidence in the eMSP
                    (certificate-issuance track record), already discounted
                    by how much the relying CPO trusts the hub's reporting
                    (see trust_pipeline.effective_emsp_confidence) -- this is
                    the actual cross-domain fusion mechanism validated in
                    the cold-start experiment, not a flat heuristic.
      - certificate_valid: a per-session, OBJECTIVE fact (hard constraint,
                    e.g. a CRL check) -- NOT subject to either authority's
                    trust score, since it is independently verifiable.

    Combination rule: MIN (weakest-link / AND-conjunction) of the two
    authority trust scores -- both the charging network's billing honesty
    AND the driver's home-network credential issuer must independently
    clear the bar; one being excellent cannot compensate for the other
    being poor. This is what "multi-authority" means operationally: neither
    score alone determines the outcome.
    """
    if not certificate_valid:
        return Decision.DENY  # objective fact, not subject to either authority's trust

    adjusted_cpo_trust = cpo_trust - (roaming_penalty if is_roaming else 0.0)
    combined_trust = min(adjusted_cpo_trust, emsp_trust)

    decision = trust_weighted_decision(1.0, combined_trust, theta_high, theta_low)
    if is_peak_hour and decision == Decision.PERMIT:
        decision = Decision.CONDITIONAL_PERMIT  # environment override, downgrade only

    return decision


In [ ]:
%%writefile benchmark_labels.py
"""
benchmark_labels.py
----------------------
Ground-truth labels shared by EVERY baseline (RBAC, ABAC, ML, DL, RL,
graph-structural) and by TrustMesh itself, so all of them are scored
against the exact same target.

GROUNDING LEDGER:
  - gt_cpo_malicious: directly from injection.py's authority_malicious
    column (1 iff cpo_id == the injected malicious CPO AND day_idx >=
    onset_day). Already a column in v2g_benchmark_multi_world.csv --
    not recomputed here, just carried through.
  - gt_emsp_malicious: ONSET-GATING FIX -- this USED to be a pure, permanent
    function of emsp_id == MALICIOUS_EMSP (identity_assignment.py's eMSP-C),
    with zero temporal variation: the transformer-baseline entity-attention
    proof of concept's own entity-only ablation confirmed this made the axis
    structurally unlearnable from behavior for ANY entity-aggregation model
    (mean- or attention-pooling), since there was never a honest-vs-malicious
    contrast in eMSP-C's own history to learn from, only its name. Now reads
    multi_world_pipeline.py/feature_engineering.py's persisted
    emsp_malicious_eligible column directly (1 iff emsp_id == MALICIOUS_EMSP
    AND day_idx >= emsp_onset_day) -- the same "identity AND onset" shape
    authority_malicious already has for CPO, so the two axes are symmetric.
    A combined CSV that predates this fix won't have that column;
    add_ground_truth_labels raises a clear error rather than silently
    falling back to the old identity-only definition. Independent threat
    axis from CPO billing fraud (independent, staggered onset day).
  - gt_ev_sybil: directly from enriched_features.py's ev_sybil column.
    Independent threat axis from both of the above.
  - risk_factor_count / composite_risk_score: NOT data fields -- these
    are EVALUATION-TARGET design choices made HERE, not anything the
    generation pipeline produces or implies. Said plainly so they can
    be cited precisely in the paper rather than mistaken for something
    collected "as-is": risk_factor_count is a simple count (0-3) of how
    many of the three independent threat axes are active for a given
    session; composite_risk_score is that count turned into [0,1] under
    an EQUAL-WEIGHTING assumption across axes (AXIS_WEIGHTS below) --
    not a claim that CPO billing fraud, eMSP credential issues, and EV
    Sybil attacks carry equal real-world severity.

WHY NOT A BAKED-IN "DECISION" COLUMN HERE: every baseline's own
continuous score gets passed through decision_engine.trust_weighted_decision()
at evaluation time (04_performance_analysis.ipynb), not pre-computed here.
That keeps the PERMIT / CONDITIONAL-PERMIT / DENY granularity a property
of each model's score, not of the ground truth -- which is what lets the
benchmark show every baseline (not just TrustMesh) supporting that same
granularity, and lets a user push to a finer granularity later just by
changing threshold cut points, with nothing in this module to revisit.
"""

import numpy as np
import pandas as pd

MALICIOUS_EMSP = "eMSP-C"  # mirrors identity_assignment.py; duplicated here
                            # deliberately so this module has no import-time
                            # dependency on trustmesh_pkg's package layout

AXIS_WEIGHTS = {"cpo": 1.0, "emsp": 1.0, "ev": 1.0}  # documented equal-weighting
                                                        # assumption, see module docstring


def add_ground_truth_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["gt_cpo_malicious"] = df["authority_malicious"].astype(int)

    if "emsp_malicious_eligible" not in df.columns:
        raise KeyError(
            "Column 'emsp_malicious_eligible' not found. This combined CSV predates "
            "the eMSP onset-gating fix (see multi_world_pipeline.py / "
            "feature_engineering.py, and this module's GROUNDING LEDGER above) -- "
            "regenerate the dataset (02_generate_v2g_benchmark.ipynb) before running "
            "this baseline. Refusing to silently fall back to the old, structurally-"
            "unlearnable identity-only definition (emsp_id == MALICIOUS_EMSP)."
        )
    df["gt_emsp_malicious"] = df["emsp_malicious_eligible"].astype(int)
    df["gt_ev_sybil"] = df["ev_sybil"].astype(int)

    df["risk_factor_count"] = (
        df["gt_cpo_malicious"] + df["gt_emsp_malicious"] + df["gt_ev_sybil"]
    )
    total_weight = sum(AXIS_WEIGHTS.values())
    df["composite_risk_score"] = (
        AXIS_WEIGHTS["cpo"] * df["gt_cpo_malicious"]
        + AXIS_WEIGHTS["emsp"] * df["gt_emsp_malicious"]
        + AXIS_WEIGHTS["ev"] * df["gt_ev_sybil"]
    ) / total_weight
    return df


def summarize_labels(df: pd.DataFrame) -> None:
    print("risk_factor_count distribution (0 = clean, 3 = all three threat axes active):")
    print(df["risk_factor_count"].value_counts().sort_index())
    print(f"\nAny risk factor present: {(df['risk_factor_count'] > 0).mean():.4f}")
    for axis in ["gt_cpo_malicious", "gt_emsp_malicious", "gt_ev_sybil"]:
        print(f"  {axis} rate: {df[axis].mean():.4f}")
    print(f"\ncomposite_risk_score describe:\n{df['composite_risk_score'].describe()}")


if __name__ == "__main__":
    import os
    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = add_ground_truth_labels(df)
    summarize_labels(df)


In [ ]:
%%writefile temporal_split.py
"""
temporal_split.py
--------------------
Train/test split for baseline evaluation: temporal within each world,
not random and not by-world.

WHY TEMPORAL: day_idx is REAL (derived from actual ACN-Data connection
timestamps, see raw_preprocess.py) and IDENTICAL in range across both
worlds -- only the synthetic identity/fraud fields differ by world, not
the real session timing. injection.py's attacks are keyed on day_idx
(onset_day), so a model that has only ever seen days before the cutoff
has a genuinely harder, more realistic job on the test split than a
model trained on a random shuffle that leaks future-day sessions into
training. This is the right split for a benchmark meant to measure "how
well/fast does each baseline detect or adapt to a trust problem," not
just "can it memorize this distribution."

Applied with the SAME day cutoff across all worlds (computed once, on
the full combined frame), since day_idx's distribution is shared real
data, not an independent per-world draw.

ONE DISCLOSED CAVEAT: enriched_features.add_baseline_charge_duration's
station+hour empirical-median fallback was computed over the WHOLE
dataset (train+test combined) when the benchmark CSV was generated, not
separately per split -- a minor leakage source for that one contextual
feature column specifically. Harmless for anything that doesn't use
baseline_charge_duration_h as a predictor of risk (it isn't one), but
worth stating plainly rather than silently ignoring, consistent with
this project's disclosure discipline.
"""

import numpy as np
import pandas as pd

TRAIN_FRACTION_OF_DAYS = 0.7  # documented split point, not tuned


def compute_day_cutoff(df: pd.DataFrame, train_fraction: float = TRAIN_FRACTION_OF_DAYS) -> int:
    days = np.sort(df["day_idx"].unique())
    cutoff_idx = int(len(days) * train_fraction)
    cutoff_idx = min(max(cutoff_idx, 1), len(days) - 1)  # keep both splits non-empty
    return int(days[cutoff_idx])


def temporal_train_test_split(df: pd.DataFrame, day_cutoff: int = None):
    if day_cutoff is None:
        day_cutoff = compute_day_cutoff(df)
    train = df[df["day_idx"] < day_cutoff].copy()
    test = df[df["day_idx"] >= day_cutoff].copy()
    print(f"[split] day_cutoff={day_cutoff}  train={len(train)} rows "
          f"(days {train['day_idx'].min()}-{train['day_idx'].max()})  "
          f"test={len(test)} rows (days {test['day_idx'].min()}-{test['day_idx'].max()})")
    for name, part in [("train", train), ("test", test)]:
        rate = (part["risk_factor_count"] > 0).mean() if "risk_factor_count" in part.columns else float("nan")
        print(f"  {name}: any-risk-factor rate = {rate:.4f}")
    return train, test, day_cutoff


if __name__ == "__main__":
    import os
    import benchmark_labels

    path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    df = pd.read_csv(path)
    df = benchmark_labels.add_ground_truth_labels(df)
    train, test, cutoff = temporal_train_test_split(df)
    print(f"\nWorld breakdown -- train:\n{train['world_id'].value_counts()}")
    print(f"World breakdown -- test:\n{test['world_id'].value_counts()}")


In [ ]:
%%writefile ml_features.py
"""
ml_features.py
-----------------
Feature set and target shared by the ML and DL baselines. Every input
here is something a live system would actually have at scoring time --
nothing from another baseline's output (RBAC/ABAC decisions, naive
trust scores) and nothing derived from the ground-truth labels
themselves.

TARGET: binary any_risk = (risk_factor_count > 0) -- the same "any real
threat axis active" definition RBAC/ABAC are already judged against
(rbac_baseline.summarize_rbac_blindness, abac_baseline.summarize_single_vs_multi).
Each model's predicted P(any_risk=0) becomes its continuous [0,1] trust
score, fed through decision_engine.trust_weighted_decision() for the
3-way PERMIT/CONDITIONAL-PERMIT/DENY decision -- same as every other
baseline, not reduced to binary.

FEATURES:
  - billing_deviation, latency_deviation: the two fuzzy-engine inputs
    (real-grounded on the metered side, see feature_engineering.py) --
    legitimate raw signals, not another baseline's processed output.
  - certificate_valid, is_roaming, is_peak_hour: real session attributes.
  - previous_flag_rate: ALREADY a real column (enriched_features.py) --
    a causal, prior-only running eMSP dispute rate. Using it here is
    feature engineering, not reusing ABAC's naive_emsp_trust (which is
    just 1 - this same column, renamed for ABAC's own purposes).
  - baseline_charge_duration_h, energy_rate_anomaly: real engineered
    columns from enriched_features.py.
  - charge_duration_h, full_duration_h, idle_h, avg_charge_rate_kw:
    raw session telemetry.
  - ev_session_share_so_far: NEW here -- a causal running statistic (same
    spirit as naive_trust.py's EV computation, RE-DERIVED independently
    so this module has no coupling to the ABAC baseline's internals) --
    the fraction of all sessions seen so far (across the FULL
    chronological sequence, not per-split -- see add_ev_session_share_feature's
    docstring) that belong to this EV.
  - day_idx: raw numeric context.

NOT included, deliberately: ev_id itself (1000-way high-cardinality
identity -- the EV-share feature above is the generalizable signal; the
raw ID would let the model memorize specific train-period EV identities
rather than learn the actual reuse-pattern signature), every
RBAC/ABAC/naive-trust output column (would be benchmarking one baseline
against a feature built from another baseline's output, not an
independent comparison) -- AND, as of this revision, cpo_id/emsp_id too.

BUG LOG (kept here deliberately, not swept away -- same discipline as
fuzzy_trust.py's own bug log): the first version of this module DID
include cpo_id/emsp_id as one-hot categorical features. On the real
125,910-row benchmark this produced AUROC/AUPRC = 1.0000 EXACTLY for
both logistic regression and random forest -- not a good score, a tell.
Root cause: in this dataset the malicious CPO/eMSP are malicious from
day 0 with no time variation (onset_day=0), so cpo_id=="CPO-B" implies
gt_cpo_malicious=1 with zero exceptions, same for emsp_id=="eMSP-C" --
a model with those IDs as input doesn't need billing_deviation or
previous_flag_rate at all; it just memorizes "if name is CPO-B or
eMSP-C, flag it," which is a lookup table, not a behavioral detector,
and wouldn't generalize to a deployment with different CPO/eMSP names.
(A synthetic smoke test run before the real data was available had the
same confound baked into its own generator, which is why it showed a
suspiciously-high-but-not-quite-1.0 AUROC~0.995 instead of flagging the
problem outright -- the real data's exact 1.0000 was the actual tell.)
Fixed by dropping cpo_id/emsp_id entirely: every remaining feature is a
genuine behavioral/contextual signal (billing_deviation,
previous_flag_rate, ev_session_share_so_far, etc.), so whatever
separability the model achieves now reflects learning from behavior,
not memorizing which specific name is the known bad one -- the same
standard TrustMesh itself is held to (its trust score comes from
billing_deviation observations, not from knowing a name in advance).

SECOND BUG LOG ENTRY (found while wiring up the 7-build variant suite,
confirmed PRE-EXISTING -- present even for the original single-build,
2-world "main" dataset): add_ev_session_share_feature's running count
and its denominator (total_seen) were kept across the ENTIRE passed-in
frame with no awareness of which world or variant a row came from --
the same class of cross-world leak documented in naive_trust.py's own
BUG LOG (world 0 and world 1, and now all 7 variants, share the same
real connection_time values, so sorting the union together interleaves
independent synthetic universes instead of treating each as its own
clean timeline). Fixed the same way: grouped by (variant, world_id)
before computing the running share, with a no-grouping fallback when
those columns aren't present."""

import numpy as np
import pandas as pd

NUMERIC_FEATURES = [
    "billing_deviation", "latency_deviation", "previous_flag_rate",
    "baseline_charge_duration_h", "energy_rate_anomaly",
    "charge_duration_h", "full_duration_h", "idle_h", "avg_charge_rate_kw",
    "ev_session_share_so_far", "day_idx",
]
BOOLEAN_FEATURES = ["certificate_valid", "is_roaming", "is_peak_hour"]
ALL_FEATURES = NUMERIC_FEATURES + BOOLEAN_FEATURES
TARGET = "any_risk"


DEFAULT_GROUP_COLS = ("variant", "world_id")  # see SECOND BUG LOG entry above


def add_ev_session_share_feature(df: pd.DataFrame, group_cols=DEFAULT_GROUP_COLS) -> pd.DataFrame:
    """Causal running per-EV session share. MUST be called on the FULL
    chronological sequence (train+test combined) BEFORE any train/test
    split -- calling it separately on an already-split test frame would
    reset every EV's running count to zero at the split boundary,
    understating exactly the reuse signature this feature exists to
    capture. (Independent of naive_trust.py's analogous computation,
    deliberately re-derived here so the ML feature set has no coupling
    to the ABAC baseline's internals.)

    Computed independently within each (variant, world_id) group when
    those columns are present (falls back to the whole frame as one
    group otherwise) -- see this module's SECOND BUG LOG entry."""
    df = df.copy()
    present = [c for c in group_cols if c in df.columns]
    groups = df.groupby(present, sort=False) if present else [(None, df)]

    share_col = pd.Series(np.nan, index=df.index, dtype=float)
    for _, g in groups:
        order = g.sort_values("connection_time").index
        ev = g.loc[order, "ev_id"].to_numpy()

        running_n: dict = {}
        total_seen = 0
        local = np.empty(len(g))
        for i, e in enumerate(ev):
            this_ev_n = running_n.get(e, 0)
            local[i] = (this_ev_n / total_seen) if total_seen > 0 else 0.0
            running_n[e] = this_ev_n + 1
            total_seen += 1
        share_col.loc[order] = local

    df["ev_session_share_so_far"] = share_col
    return df


def add_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[TARGET] = (df["risk_factor_count"] > 0).astype(int)
    return df


def prepare_ml_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Adds the EV-share feature and target, and coerces the boolean
    columns to int (sklearn-friendly). Call on the FULL combined
    chronological sequence, before splitting -- see
    add_ev_session_share_feature's docstring."""
    df = add_ev_session_share_feature(df)
    df = add_target(df)
    for col in BOOLEAN_FEATURES:
        df[col] = df[col].astype(int)
    return df


## Step 2: clear stale checkpoints, then write and run the ML baseline

If you ran an earlier version of this notebook, `ml_baseline/` on your Drive already has `.joblib` files from it. Clearing the whole directory is REQUIRED here, for two independent reasons, not just the original cpo_id/emsp_id fix: (1) the model filenames (`logistic_regression.joblib`, `random_forest.joblib`) haven't changed, but what they MEAN has -- an old file holds a model trained on all 7 variants pooled together; this version's checkpoint check would otherwise silently load that stale, wrongly-scoped model instead of retraining on `main` only; (2) `ml_features.py`'s cross-(variant, world_id) running-statistic fix (see this notebook's intro) changes the `ev_session_share_so_far` feature's values, so a model trained before the fix was fit on subtly wrong inputs either way.

In [ ]:
import os
import shutil, os
ml_out_dir = os.environ.get('V2G_ML_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/ml_baseline')
if os.path.isdir(ml_out_dir):
    shutil.rmtree(ml_out_dir)
    print(f'Cleared {ml_out_dir}')
else:
    print('Nothing to clear.')

In [ ]:
%%writefile ml_baseline.py
"""
ml_baseline.py
----------------
Classical ML baseline: trains two model families (logistic regression,
random forest) on ml_features.ALL_FEATURES against the binary any_risk
target. Each produces a continuous [0,1] trust score
(1 - P(any_risk=1)), passed through decision_engine.trust_weighted_decision()
for the SAME 3-way PERMIT/CONDITIONAL-PERMIT/DENY decision space as
every other baseline -- not reduced to binary.

CPU-bound (sklearn; no GPU benefit at this feature/row count). Trained
SEQUENTIALLY, not via multiprocessing.Pool, deliberately -- a local
smoke test on synthetic data caught a real bug in the Pool-of-models
design this module started with: multiprocessing.Pool workers are
daemonic processes, and a daemonic process is NOT ALLOWED to spawn its
own children, so RandomForestClassifier(n_jobs=-1)'s internal tree-level
parallelism was being silently forced back to n_jobs=1 inside a Pool
worker (joblib/loky prints "Loky-backed parallel loops cannot be called
in a multiprocessing" and degrades rather than erroring, which is what
made it easy to miss). LogisticRegression's own n_jobs is a no-op here
anyway (it only parallelizes one-vs-rest for >2 classes; this is binary).
So the real, higher-impact parallelism lever for THIS baseline is
RandomForest's n_jobs=-1 running directly in the main process, not a
model-level Pool fighting it for the same cores -- the opposite of
multi_world_pipeline.py's 2-worlds case, where the work inside each
worker had no parallelism of its own to lose.

LOADS THE COMBINED CSV, NOT THE PRE-SPLIT train/test FILES: ev_session_share_so_far
is a causal running statistic and must be computed over the FULL
chronological sequence before splitting (see ml_features.py's docstring)
-- reloading and re-deriving the split here, identically to
03_prepare_baseline_labels_splits.ipynb (a pure function of the same
inputs, so it reproduces the exact same day_cutoff/rows), is what keeps
that correct. This mirrors why 04_rbac_abac_baselines.ipynb also reloaded
the combined CSV rather than the pre-split files.

VARIANT-AWARE EVALUATION (train once on `main`, zero-shot elsewhere):
mirrors the reference ACCESS-IoT dataset's own Table 5 precedent --
each model is trained ONCE, on `main`'s train split only, then the SAME
frozen model (no retraining) is scored against every build's OWN test
split (main, sparse_feedback, noisy_benign, adversarial, seed_101,
seed_103, seed_104). This is a deliberate zero-shot generalization
test, not 7 independent from-scratch experiments: it asks "does a
model trained under ordinary conditions hold up when evaluated under a
quieter audit trail, noisier honest behavior, or a camouflaged
attacker?" -- not "can a model be retrained to beat each variant." Needs
ml_features.add_ev_session_share_feature's cross-(variant, world_id)
grouping fix (see ml_features.py's SECOND BUG LOG entry) to be safe to
compute over all 7 variants' worlds at once.

CHECKPOINTING: each model's artifact (joblib), trained on `main` only,
is written the moment training finishes; re-running skips training for
any model whose checkpoint already exists. The per-variant evaluation
table itself is cheap (scoring, not training) and is always
recomputed/overwritten, so adding a variant later never leaves a stale
summary row behind.

BUG LOG: the first version's pipelines one-hot encoded cpo_id/emsp_id as
features via a ColumnTransformer. On the real benchmark that produced
AUROC/AUPRC = 1.0000 exactly -- see ml_features.py's own bug log for the
root cause (identity trivially determines the label in this dataset,
since the malicious CPO/eMSP are malicious from day 0 with no time
variation). Fixed by dropping ml_features.CATEGORICAL_FEATURES
entirely (the constant no longer exists) and simplifying both pipelines
to a single numeric+boolean preprocessing step -- no ColumnTransformer
branch needed anymore since there's nothing categorical left to encode.

SECOND BUG LOG ENTRY: the original version trained and evaluated on the
FULL POOLED combined dataframe (all 7 variants mixed into one
train/test split), inheriting ml_features.py's cross-(variant,
world_id) running-statistic leak and conflating "how good is the model"
with "how do the 7 deliberately-different builds compare" into one
number. Restructured into the train-on-main/zero-shot-eval-everywhere
design described above once the variant suite existed.
"""

import os
import sys
import time

import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

sys.path.insert(0, os.path.dirname(__file__) or ".")

import ml_features
import benchmark_labels
import temporal_split
from decision_engine import trust_weighted_decision

OUT_DIR = os.environ.get(
    "V2G_ML_OUT_DIR",
    "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/ml_baseline",
)


def _make_logistic_regression():
    return Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])


def _make_random_forest():
    # Trees don't need scaling -- plain pipeline, no preprocessing step.
    return Pipeline([
        ("clf", RandomForestClassifier(n_estimators=200, max_depth=12, n_jobs=-1,
                                         class_weight="balanced", random_state=42)),
    ])


MODEL_BUILDERS = {
    "logistic_regression": _make_logistic_regression,
    "random_forest": _make_random_forest,
}


TRAIN_VARIANT = "main"  # train exclusively on this build; every other
                          # build is evaluated zero-shot (see module docstring)


def model_checkpoint_path(model_name):
    return os.path.join(OUT_DIR, f"{model_name}.joblib")


def variant_preds_path(model_name, variant_name):
    return os.path.join(OUT_DIR, f"{model_name}_{variant_name}_test_predictions.csv")


def _prepare_full_frame(combined_csv_path):
    df = pd.read_csv(combined_csv_path)
    df["connection_time"] = pd.to_datetime(df["connection_time"], utc=True)
    df = benchmark_labels.add_ground_truth_labels(df)
    # causal EV-share feature, computed per (variant, world_id) group over
    # each group's own FULL chronological sequence (see ml_features.py's
    # SECOND BUG LOG entry) -- safe to call on all 7 variants' worlds at once.
    df = ml_features.prepare_ml_frame(df)
    day_cutoff = temporal_split.compute_day_cutoff(df)
    return df, day_cutoff


def get_or_train_model(model_name, df, day_cutoff):
    """Loads model_name's joblib checkpoint if one already exists; otherwise
    trains it ONCE on TRAIN_VARIANT's ("main") train split only and saves
    it. Runs in the main process, sequentially -- see module docstring for
    why this is NOT wrapped in multiprocessing.Pool. RandomForest's own
    n_jobs=-1 (set in _make_random_forest) does the real parallel work."""
    model_path = model_checkpoint_path(model_name)
    if os.path.exists(model_path):
        print(f"[{model_name}] model checkpoint already exists -- loading, not retraining")
        return joblib.load(model_path)

    t0 = time.time()
    train_df = df[(df["variant"] == TRAIN_VARIANT) & (df["day_idx"] < day_cutoff)]
    X_train, y_train = train_df[ml_features.ALL_FEATURES], train_df[ml_features.TARGET]

    pipeline = MODEL_BUILDERS[model_name]()
    pipeline.fit(X_train, y_train)

    os.makedirs(OUT_DIR, exist_ok=True)
    joblib.dump(pipeline, model_path)
    print(f"[{model_name}] trained on '{TRAIN_VARIANT}' only: n_train={len(train_df)}  "
          f"({time.time() - t0:.1f}s) -> {model_path}")
    return pipeline


def evaluate_on_variant(model_name, pipeline, df, day_cutoff, variant_name):
    """Scores the ALREADY-TRAINED (on 'main' only) pipeline against
    variant_name's OWN test split -- zero-shot, no retraining. This is
    the per-variant row in the Table-5-style summary."""
    test_df = df[(df["variant"] == variant_name) & (df["day_idx"] >= day_cutoff)]
    X_test, y_test = test_df[ml_features.ALL_FEATURES], test_df[ml_features.TARGET]

    proba_risk = pipeline.predict_proba(X_test)[:, 1]
    trust_score = 1.0 - proba_risk

    auroc = roc_auc_score(y_test, proba_risk)
    auprc = average_precision_score(y_test, proba_risk)
    decisions = [trust_weighted_decision(1.0, float(t)).value for t in trust_score]

    preds_path = variant_preds_path(model_name, variant_name)
    out = test_df[["session_id", "variant", "cpo_id", "emsp_id", "ev_id", "day_idx",
                    "risk_factor_count", "any_risk"]].copy()
    out["trust_score"] = trust_score
    out["decision"] = decisions
    out.to_csv(preds_path, index=False)

    return {"model": model_name, "variant": variant_name,
            "auroc": float(auroc), "auprc": float(auprc), "n_test": int(len(test_df))}


def main(combined_csv_path):
    model_names = list(MODEL_BUILDERS.keys())
    df, day_cutoff = _prepare_full_frame(combined_csv_path)
    variants = sorted(df["variant"].unique()) if "variant" in df.columns else [None]
    print(f"day_cutoff={day_cutoff}  variants found: {variants}")
    print(f"Training {len(model_names)} model(s) sequentially on '{TRAIN_VARIANT}' only, "
          f"then evaluating each zero-shot on every variant's own test split "
          f"(RandomForest uses n_jobs=-1 internally; see module docstring)")

    all_rows = []
    for model_name in model_names:
        pipeline = get_or_train_model(model_name, df, day_cutoff)
        for variant_name in variants:
            row = evaluate_on_variant(model_name, pipeline, df, day_cutoff, variant_name)
            all_rows.append(row)
            print(f"  [{model_name:20s} -> {variant_name:16s}] "
                  f"AUROC={row['auroc']:.4f}  AUPRC={row['auprc']:.4f}  n_test={row['n_test']}")

    summary = pd.DataFrame(all_rows)
    summary_path = os.path.join(OUT_DIR, "ml_baseline_variant_summary.csv")
    os.makedirs(OUT_DIR, exist_ok=True)
    summary.to_csv(summary_path, index=False)

    print(f"\n=== Summary table (trained once on '{TRAIN_VARIANT}', zero-shot evaluated "
          f"per variant) -- saved to {summary_path} ===")
    print(summary.pivot(index="variant", columns="model", values="auroc")
          .round(4).to_string())

    seed_variants = [v for v in variants if v.startswith("seed_")]
    if seed_variants:
        print(f"\n=== Stability check across {len(seed_variants)} independent reseeds "
              f"({', '.join(seed_variants)}) -- mean +/- std AUROC per model ===")
        stab = summary[summary["variant"].isin(seed_variants)].groupby("model")["auroc"]
        print(stab.agg(["mean", "std"]).round(4).to_string())

    return summary


if __name__ == "__main__":
    combined_csv_path = os.environ.get(
        "V2G_COMBINED_CSV",
        "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv",
    )
    main(combined_csv_path)


In [ ]:
import os
import ml_baseline

COMBINED_CSV = os.environ.get('V2G_COMBINED_CSV', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv')
summary = ml_baseline.main(COMBINED_CSV)

## Step 3: insights -- AUROC per variant, per model

One chart, reusing `ml_baseline_variant_summary.csv` from Step 2: how each frozen, `main`-trained model's AUROC holds up across the other 6 builds. Expect `sparse_feedback`/`noisy_benign` to barely move it (by design -- they change how OFTEN/how noisily a signal is observed, not its underlying separability) and `adversarial` to be the one build that visibly drops it (camouflaged fraud is the hardest case for a model that never saw it during training) -- the same pattern the reference ACCESS-IoT paper reports for its own analogous variants.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

ML_OUT_DIR = os.environ.get('V2G_ML_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/ml_baseline')
summary = pd.read_csv(os.path.join(ML_OUT_DIR, 'ml_baseline_variant_summary.csv'))
pivot = summary.pivot(index='variant', columns='model', values='auroc')
print(pivot.round(4))

ax = pivot.plot(kind='bar', figsize=(9, 5), ylim=(0, 1.05), legend=False)
ax.set_ylabel('AUROC (test split, zero-shot per variant)')
ax.set_title("ML baseline: trained once on 'main', evaluated zero-shot per variant")
ax.axhline(1.0, color='gray', linewidth=0.5, linestyle='--')
ax.legend(title='model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('ml_baseline_variant_insights.pdf', dpi=300)
plt.show()
print('Saved ml_baseline_variant_insights.png')

## Step 4: beyond AUROC/AUPRC -- F1, precision/recall, confusion matrix, per-axis breakdown

AUROC/AUPRC alone can look deceptively perfect: `main`/`seed_101`/`seed_103`/`seed_104` sit at or near 1.0000 because, with no camouflage active, `billing_deviation` and `previous_flag_rate` are each ALREADY near-perfectly separating single features for their own axis (confirmed back in `02_generate_v2g_benchmark.ipynb`'s data-insights cell) -- `any_risk` is just the OR of three mostly-solved sub-problems, so a model combining them saturates easily. That's an expected property of the unCamouflaged builds (and matches the reference ACCESS-IoT paper's own pattern), not automatically a leak -- but an exact 1.0000 deserves scrutiny by this project's own standard (see `ml_features.py`'s first BUG LOG entry), so this step adds the metrics to actually check: F1/precision/recall/accuracy at the standard 0.5 probability cutoff, a confusion matrix, Brier score (calibration), F1 against the ACTUAL operational decision (DENY/CONDITIONAL-PERMIT vs PERMIT, not just the 0.5 cutoff), a full `sklearn.classification_report`, and -- the most diagnostic piece -- whether the SAME continuous score also separates each INDIVIDUAL threat axis (`gt_cpo_malicious`, `gt_emsp_malicious`, `gt_ev_sybil`) on its own, not just the pooled `any_risk` target. A low per-axis AUROC on one axis hiding behind a high pooled AUROC is exactly the kind of thing the pooled metric alone would never surface.

Reuses the per-variant prediction CSVs Step 2 already saved -- no retraining, just re-deriving the per-axis ground truth from the combined CSV (a pure function of columns already in it) and joining by `session_id`.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                              confusion_matrix, classification_report, roc_auc_score,
                              average_precision_score, brier_score_loss)
import benchmark_labels

OUT_DIR = os.environ.get('V2G_ML_OUT_DIR', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/ml_baseline')
COMBINED_CSV = os.environ.get('V2G_COMBINED_CSV', '/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds/v2g_benchmark_multi_world.csv')
MODELS = ['logistic_regression', 'random_forest']
DECISION_THRESHOLD = 0.5  # standard probability cutoff for the metrics below;
                           # independent of decision_engine's theta_high/theta_low,
                           # which drive the PERMIT/CONDITIONAL-PERMIT/DENY column instead

# Added 'emsp_malicious_eligible' to the loaded columns so benchmark_labels can calculate 'gt_emsp_malicious'
labels_df = pd.read_csv(COMBINED_CSV, usecols=['session_id', 'variant', 'authority_malicious', 'emsp_id', 'ev_sybil', 'emsp_malicious_eligible'])
labels_df = benchmark_labels.add_ground_truth_labels(labels_df)
labels_df = labels_df[['session_id', 'gt_cpo_malicious', 'gt_emsp_malicious', 'gt_ev_sybil']]

import glob, os as _os
# Strip the KNOWN model-name prefix/suffix rather than splitting on '_' --
# 'logistic_regression' itself contains an underscore, so a naive split
# would cut the variant name in the wrong place.
_prefix, _suffix = f'{MODELS[0]}_', '_test_predictions.csv'
variants = sorted({
    _os.path.basename(p)[len(_prefix):-len(_suffix)]
    for p in glob.glob(_os.path.join(OUT_DIR, f'{_prefix}*{_suffix}'))
})
print(f'variants found from saved prediction files: {variants}')

overall_rows, axis_rows = [], []
for model in MODELS:
    for variant in variants:
        preds = pd.read_csv(f'{OUT_DIR}/{model}_{variant}_test_predictions.csv')
        preds = preds.merge(labels_df, on='session_id', how='left')

        y_true = preds['any_risk'].astype(int).to_numpy()
        proba_risk = 1.0 - preds['trust_score'].to_numpy()
        y_pred = (proba_risk >= DECISION_THRESHOLD).astype(int)
        flagged_by_decision = preds['decision'].isin(['DENY', 'CONDITIONAL-PERMIT']).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        overall_rows.append({
            'model': model, 'variant': variant,
            'auroc': roc_auc_score(y_true, proba_risk),
            'auprc': average_precision_score(y_true, proba_risk),
            'brier': brier_score_loss(y_true, proba_risk),
            'f1@0.5': f1_score(y_true, y_pred, zero_division=0),
            'precision@0.5': precision_score(y_true, y_pred, zero_division=0),
            'recall@0.5': recall_score(y_true, y_pred, zero_division=0),
            'accuracy@0.5': accuracy_score(y_true, y_pred),
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
            'f1_decision': f1_score(y_true, flagged_by_decision, zero_division=0),
        })

        # Per-axis: does this SAME continuous score also separate each
        # INDIVIDUAL threat axis, not just the pooled any_risk target?
        for axis in ['gt_cpo_malicious', 'gt_emsp_malicious', 'gt_ev_sybil']:
            y_axis = preds[axis].astype(int).to_numpy()
            axis_auroc = float('nan') if (y_axis.sum() == 0 or y_axis.sum() == len(y_axis)) else roc_auc_score(y_axis, proba_risk)
            axis_rows.append({'model': model, 'variant': variant, 'axis': axis, 'auroc': axis_auroc})

overall = pd.DataFrame(overall_rows)
by_axis = pd.DataFrame(axis_rows)
metrics_path = f'{OUT_DIR}/ml_baseline_variant_metrics_full.csv'
overall.to_csv(metrics_path, index=False)
print(f'Saved -> {metrics_path}\n')

for model in MODELS:
    print(f'=== {model}: overall (any_risk) metrics ===')
    print(overall[overall["model"]==model].set_index('variant')[
        ['auroc','auprc','brier','f1@0.5','precision@0.5','recall@0.5','accuracy@0.5','f1_decision']
    ].round(4).to_string())
    print()
    print(f'=== {model}: confusion matrix (tn/fp/fn/tp @ proba>=0.5) ===')
    print(overall[overall["model"]==model].set_index('variant')[['tn','fp','fn','tp']].to_string())
    print()
    print(f'=== {model}: per-axis AUROC (does the SAME score separate each individual threat axis?) ===')
    print(by_axis[by_axis["model"]==model].pivot(index='variant', columns='axis', values='auroc').round(4).to_string())
    print()

# Full sklearn classification_report, model x variant -- ready to paste into a paper appendix.
for model in MODELS:
    for variant in variants:
        preds = pd.read_csv(f'{OUT_DIR}/{model}_{variant}_test_predictions.csv')
        y_true = preds['any_risk'].astype(int).to_numpy()
        proba_risk = 1.0 - preds['trust_score'].to_numpy()
        y_pred = (proba_risk >= DECISION_THRESHOLD).astype(int)
        print(f'--- {model} / {variant} ---')
        print(classification_report(y_true, y_pred, target_names=['no_risk','any_risk'], digits=4, zero_division=0))

## Done

Each model's `.joblib` (trained on `main` only), `<model>_<variant>_test_predictions.csv` (session_id, trust score, decision, per variant), the combined `ml_baseline_variant_summary.csv` (AUROC/AUPRC per model x variant), and `ml_baseline_variant_metrics_full.csv` (F1/precision/recall/accuracy/Brier/confusion matrix per model x variant) are in `.../worlds/ml_baseline/`. Paste back the printed per-variant AUROC/AUPRC table, the stability check across the seed replicates, the saved chart, and the Step 4 metrics tables (especially the per-axis AUROC breakdown).